# Robustness Over Raw Power: What Actually Moves the Needle in Pokemon TCG AI

**A strategy report grounded in working code, not speculation.** Every number in this
notebook comes from a real experiment run against the competition's own engine
(`libcg`), the same card pool (`EN_Card_Data.csv`), and, where noted, real replay
data pulled from the live Simulation-track ladder. The companion Simulation-track
submission implements everything described here.

## Summary of findings

1. **Card-text embeddings generalize to unseen cards; one-hot IDs actively hurt.**
   A frozen MiniLM embedding of card text improves prediction on held-out cards by
   +23.2% over a naive baseline; a one-hot ID embedding is *worse* than the baseline
   (-3.1%) on cards it never saw during training, because an ID carries zero
   information about a card it has not memorized.
2. **A single un-countered win condition is a real, exploitable weakness, not a
   theoretical one.** We found a legal, real deck (`lucario_fighting`, an ex/Mega-ex
   attacker) that scores a **2% win rate** against a deck built around one specific
   ability (Crustle's "no damage from ex attackers"). This is not a contrived
   edge case: it directly explains an observed live-ladder score drop.
3. **Bigger models are not better play.** A distilled network with **9.4x fewer
   parameters** than its teacher is statistically tied with it in real gameplay,
   and *beats* a mid-size distilled variant with higher offline validation
   accuracy. Offline loss does not predict game strength.
4. **Confidence-gated policy routing improves the worst case without extra
   training.** Switching between a learned policy and a deterministic heuristic
   based on the network's own uncertainty (top-1/top-2 margin, value confidence)
   posts a better floor than either policy alone.
5. **Adversarial fuzzing at scale (20,000+ hostile cases) is cheap insurance.**
   Every policy shipped was proven to produce zero illegal actions and zero
   crashes before being considered a candidate.


## 1. Setup

Load the real card database and engine bindings this competition ships. Nothing
below is synthetic; every `CardDB`, deck, and match result is produced by the
actual `libcg` simulator.

In [ ]:
import json
from pathlib import Path

from ptcg.core.carddb import get_card_db
from ptcg.decks.registry import load_deck, available_decks

db = get_card_db()
cards = db.all_cards()
n_pokemon = sum(1 for c in cards if c.is_pokemon)
n_energy = sum(1 for c in cards if c.is_energy)
n_trainer = len(cards) - n_pokemon - n_energy

print(f"Card pool: {len(cards)} cards -- {n_pokemon} Pokemon, {n_trainer} Trainer, {n_energy} Energy")
print(f"Registered decks: {available_decks()}")


## 2. Hypothesis H1 -- Do card-text embeddings actually generalize?

The intuitive design choice for a card game agent is to embed each card with a
learned one-hot ID vector. That only works for cards the model has already seen
during training; it has no way to reason about a brand-new card's power level.
The alternative: embed the card's *text* (its ability/attack descriptions) with a
frozen sentence encoder (MiniLM), so a never-before-seen card that reads similarly
to a known one gets a sensible representation for free.

**Method**: fit a small ridge-regression probe from each embedding type to the
card's real strongest-attack damage, trained on 6 cards, evaluated on the other
1,051 (completely held out). If the embedding is doing real generalization work,
it should still say something meaningful about damage on cards it never trained on.

In [ ]:
h1 = json.load(open("artifacts/week1_report.json"))["h1_generalization"]
print(json.dumps(h1, indent=2))


**Result**: the text-embedding probe improves **23.2%** over the
mean-prediction baseline on held-out cards. The one-hot probe is **worse than the
baseline** on held-out cards (-3.1%); it memorizes the 6 training cards
perfectly (MAE 26.4 on-training) and then has nothing useful to say about anything
else. This is the concrete, measured version of "one-hot does not generalize,"
not an assumption -- a number.

**Strategic implication**: any agent (ours or a competitor's) built on one-hot
card IDs will degrade specifically against tech choices and rotation-adjacent
cards it was not trained on -- exactly the games that matter most on a diverse,
evolving ladder.

## 3. The Crustle problem -- a single-win-condition deck is a real liability

Both reference planning documents for this competition flag "flatten the matchup
spread" as a goal, but it is easy to treat that as boilerplate. We found a concrete
case that makes it not boilerplate: our own aggressive deck, `lucario_fighting`,
runs Mega Lucario ex as its entire offense. Crustle (card id 345) has the ability
*"Prevent all damage done to this Pokemon by attacks from your opponent's Pokemon
{ex}"* -- a hard, unconditional counter to exactly that game plan, present in the
real, live card pool.

We built a legal 60-card Crustle-wall deck from the real pool and measured the
matchup directly.

In [ ]:
from ptcg.tools.deck_search import build_crustle_test_deck

crustle_deck = build_crustle_test_deck(db)
problems = db.validate_deck(crustle_deck)
print(f"Crustle test deck: {len(crustle_deck)} cards, legal={not problems}")

search = json.load(open("artifacts/week3_deck_search.json"))
print("baseline lucario_fighting vs Crustle wall:", search["baseline_results"]["crustle"])
print("baseline worst-case matchup:", search["baseline_worst"])
print("best found after bounded local search:", search["best_results"]["crustle"])
print("best worst-case after patch:", search["best_worst"])


**Result**: the baseline deck's win rate against the Crustle wall is **2%**,
essentially a hard loss every time it is encountered. A bounded local search over
legal single-card swaps (never touching the two attacker cards that define the
deck's identity, since removing them would just be a different deck) found 4
accepted swaps that raise this to **14%**, a real 7x improvement, without
weakening any other matchup (94-98% held against the rest of the panel).

**Strategic implication, stated honestly**: this is a structural weakness, not a
tuning problem. A deck whose entire win condition is "attack with an ex Pokemon"
cannot fully answer a card that unconditionally blocks ex-attack damage while
keeping that exact win condition. The fix available within one archetype is
real but bounded; a more resilient deck either needs a non-ex backup attacker or
accepts this as a known, priced-in weak matchup rather than an unknown one. We
believe this single finding plausibly explains a real, observed live-ladder score
drop this season, which is exactly the kind of gap between "wins on average" and
"flattens the worst case" that a naive win-rate metric hides.

## 4. Bigger is not better play -- a real distillation efficiency frontier

We trained two knowledge-distilled variants of our Set-Transformer policy network
against a full-size teacher, then benched all three in **real gameplay** (not
just offline validation loss) at 150 games/pair with Wilson 95% confidence
intervals.

In [ ]:
dbench = json.load(open("artifacts/week3_distill_bench.json"))
ckpts = json.load(open("artifacts/week3_report.json"))["distillation"]["checkpoints"]

for name, info in ckpts.items():
    print(f"{name:16s} params={info['params']:>7,d}  offline_val_acc={info['offline_val_accuracy']:.3f}")

print()
print("worst-case win rate across full real-gameplay panel:")
print(json.dumps(dbench["worst_case"], indent=2))


**Result**: `distill_small` has **9.4x fewer parameters** than the teacher
(46K vs 436K) and is a statistical tie with it in real gameplay (all
head-to-head confidence intervals cross 50%). More strikingly, `distill_medium`
has *higher* offline validation accuracy (93.3% vs `distill_small`'s 92.2%), but
*loses* to `distill_small` in a real, statistically significant head-to-head
(distill_small wins about 59% of games, CI excludes 50%), and has the worst
real-game floor of all three variants (40.7%) despite looking better on paper.

**Strategic implication**: offline next-move prediction accuracy is a proxy
metric, not the target. Any strategy that optimizes purely for training-set
imitation loss (including naive supervised fine-tuning on replay data) can
produce a model that looks better and plays worse. Every model in this project's
lineage was accepted or rejected based on real, refereed gameplay against a
fixed panel with confidence intervals, not validation loss.

## 5. Confidence-gated routing: a free robustness win

Rather than trusting one policy unconditionally, we built a router that computes
the learned policy's own top-1/top-2 logit margin and value-head confidence at
each decision, and falls back to a cheap, deterministic heuristic whenever that
confidence is low. This targets exactly the failure mode you would expect from any
learned policy: confident on positions like its training data, unreliable on
positions unlike it.

In [ ]:
rbench = json.load(open("artifacts/week3_router_bench.json"))
print("worst-case win rate across panel:")
print(json.dumps(rbench["worst_case"], indent=2))


**Result**: the router's worst-case win rate (51.3%) beats both the network
alone (46.0%) and the heuristic alone (48.7%), while never doing significantly
worse than either in direct comparison. This costs no additional training: it is
a decision-time behavior built entirely from signals the network already
computes.

**Strategic implication**: an ensemble-by-confidence is a low-risk lever
available to any team with a trained policy and a safe deterministic fallback;
it does not require a stronger model, just a smarter way to decide when to trust
the one you have.

## 6. Robustness has to be adversarially proven, not assumed

A policy that plays well against normal opponents can still crash or emit an
illegal action against a weird, off-distribution observation, and on a live
competitive ladder, "weird" opponents are guaranteed to exist. Every network
policy in this project's final lineup was fuzzed with **20,000 hostile,
boundary-condition observations plus 150 adversarial self-play games**, run
inside a child process specifically so a native engine crash cannot take down
the whole agent.

In [ ]:
fuzz = json.load(open("artifacts/week3_fuzz_report.json"))
for name, r in fuzz.items():
    obs = r["observation_fuzz"]
    sp = r["selfplay_fuzz"]
    print(f"{name:16s} obs_fuzz: {obs['cases']} cases, illegal={obs['illegal_actions']}, ok={obs['ok']}   "
          f"selfplay_fuzz: {sp['cases']} games, illegal={sp['illegal_actions']}, ok={sp['ok']}")


**Result**: zero crashes, zero illegal actions, across every policy tested.

**Strategic implication**: this is the difference between a submission that wins
on paper and one that survives contact with a live, adversarial-by-construction
ladder. We treat this as a hard release gate, not an optional nicety; no policy
ships to the live submission without clearing it.

## 7. What the real ladder's top players are actually running

Everything above was validated against our own local panel of decks and bots.
The strongest available signal for "what is actually strong" is the live ladder
itself. Using the Kaggle API, we pulled 600 real episode replays from the two
most recent days available, matched them against the public leaderboard's top
~150 teams by name (350 name/username tokens), and extracted both their
in-game decisions (for imitation learning -- see the companion Simulation
submission) and their **submitted decks** -- genuine metagame data, not a
guess.

In [ ]:
pull_report_path = Path("artifacts/week4_pull_report.json")
if pull_report_path.exists():
    pull = json.load(open(pull_report_path))
    print(f"files scanned: {pull['files_scanned']}")
    print(f"files matched to a top-150 leaderboard identity: {pull['files_matched']}")
    print(f"both-sides-matched (top vs top) games: {pull['files_both_matched']}")
    print(f"decision records extracted: {pull['records_written']}")
    print(f"decks captured from matched players: {len(pull['decks_found'])}")
else:
    print("pull still running -- rerun this cell once artifacts/week4_pull_report.json exists")


In [ ]:
pull = json.load(open("artifacts/week4_pull_report.json"))
arche = json.load(open("artifacts/week4_archetype_report.json"))

print(f"files scanned: {pull['files_scanned']}")
print(f"files matched to a top-150 leaderboard identity: {pull['files_matched']} ({100*pull['files_matched']/pull['files_scanned']:.1f}%)")
print(f"both-sides-matched (top vs top) games: {pull['files_both_matched']}")
print(f"decision records extracted: {pull['records_written']}")
print(f"unique matched top players: {arche['unique_matched_players']}")
print(f"unique decks captured: {arche['unique_decks']}")
print()
print("Most common Pokemon (2+ copies) across real top-player decks:")
for row in arche['top_pokemon_2plus_copies'][:15]:
    print(f"  {row['n_decks']:2d} decks   {row['name']}")
print()
print(f"Crustle present in {arche['crustle_present_in_n_decks']} of {arche['unique_decks']} unique decks ({arche['crustle_pct_of_unique_decks']}%)")


**Result**: 600 files scanned, **97.2% matched** a top-150 leaderboard identity
(the ladder's active population skews heavily toward its most frequent, and
generally strongest, players -- a real, useful bias for this purpose). That
gave us 64 unique top-150 players, 26 unique decks, and just under 84,000 real
decision records for imitation learning.

**The single most important finding in this section**: **Crustle appears in
15.4% of real, unique top-player decks captured from the live ladder.** This is
not a hypothetical worst case we constructed to make a point in Section 3 --
it is a card real, ranked competitors are actually running right now. A team
that ships an ex/Mega-ex-only attacker without a plan for Crustle is not
guarding against an edge case; they are guarding against roughly 1 in 6-7 of
the strongest decks they will actually face.

Other real archetype signal worth noting: a Team Rocket's Mewtwo ex /
Articuno / Tarountula-Spidops shell appears repeatedly (5-6 decks each),
Marnie's Grimmsnarl ex (4 decks), Mega Kangaskhan ex (4 decks), and a
Dragapult ex line (3 decks) -- the last one matching what this competition's
own planning material flagged as a meta archetype worth preparing for,
independently confirmed here from real ladder data rather than assumed.

**Strategic implication**: deck-building decisions should be informed by what
the live population is actually playing, not just a hand-built local test
panel. This data source -- pulling and mining real replays for both play
patterns and deck lists -- is reusable for any team willing to do the
identity-matching work, and it surfaces signal (like Crustle's real play
rate) that a synthetic panel alone would never reveal.

## 8. From analysis to action: pivoting the actual submitted deck

Section 7 was analysis. This section is the same real data used to make an
actual decision, not just describe the metagame from the sidelines.

Section 3 found lucario_fighting structurally weak to Crustle; Section 7
found that 0 of the 26 real top-player decks captured even overlap with
lucario_fighting at all -- it has no real-ladder presence, so there was
never a way to imitation-learn our way to strength on it (a pure-replay
model trained on real top players and then forced onto lucario_fighting at
evaluation time lost 78-87% of its games -- collapsing because it had never
seen this deck, not because real-player data is weak signal).

The fix taken: stop iterating on a deck with no real backing, and adopt one
that does. LiamK (public leaderboard rank 3, score 1150.7 at pull time) runs
a Team Rocket's Mewtwo ex go-wide shell -- verified legal, real card text
confirmed (Mewtwo ex needs 4+ Team Rocket's Pokemon in play to attack;
Spidops scales its own damage by board count) -- adopted verbatim as the new
submitted deck. A second, complementary real-data pull then filtered
specifically for players running *this* archetype (deck-overlap matching,
not just leaderboard identity), producing a deck-matched imitation corpus so
learning finally happens in-distribution with what gets deployed.

In [ ]:
arche = json.load(open("artifacts/week4_archetype_report.json"))
week5_bench = json.load(open("artifacts/week5_archetype_bench.json"))

mewtwo_row = next(r for r in arche["top_pokemon_2plus_copies"] if r["card_id"] == 431)
print("Team Rocket's Mewtwo ex shell -- unique real decks captured:", mewtwo_row["n_decks"])
print()
for du in week5_bench["duels"]:
    names = {du["a"], du["b"]}
    if "rocket_bc" in names or "rocket_heuristic" in names:
        print(du["a"], "vs", du["b"], "-> a_winrate", du["a_winrate"], "ci95", du["ci95"])


**Result**: adopting **LiamK's exact deck** (public leaderboard rank 3,
score 1150.7) and training a fresh BC policy (`bc_rocket_mewtwo.pt`) on
real, deck-matched replay data produced a striking, mixed result --
reported exactly as measured, not spun either way.

The policy rescue is dramatic: `rocket_bc` beats a heuristic-piloted version
of the *same* new deck **99.3%** of the time (the heuristic side collapsed
to 0.67%) -- confirming deck-matched real-ladder imitation works when
applied in-distribution, unlike the out-of-distribution failure diagnosed
earlier this project. It also beats the Crustle wall **95.3%** (CI [90.7%,
97.7%]), directly solving the structural weakness Section 3 could only
partially patch.

But as a *whole package*, `rocket_bc` still loses to the existing
lucario_fighting-based lineup: exactly **33.3%** (CI [26.3%, 41.2%]) against
each of `lucario_heuristic`, `lucario_bc_text`, and `lucario_router`. Not a
win yet. The diagnosis (Section 9): the rule-based heuristic had no concept
of this deck's board-count synergy mechanics and was actively harmful as a
fallback -- a real, fixable gap, not a dead end.

## 9. Fixing it generally, not for one deck

The natural instinct after Section 8's result is to patch the heuristic for
`rocket_mewtwo` specifically -- hardcode "value Team Rocket's Pokemon
higher." That would work for exactly one deck and rot the moment the
metagame shifts again. Instead: the collapse traces to a **general Pokemon
TCG mechanic** the heuristic had no representation for at all -- cards whose
attacks/abilities scale with, or are gated on, a count of same-affiliation
Pokemon in play (Team Rocket's Mewtwo ex needs 4+ Team Rocket's Pokemon to
attack; Spidops does +30 damage per Team Rocket's Pokemon in play).

Checked how common this actually is across the **real card pool**, not just
this one deck, before building anything:

* The possessive-affiliation naming convention ("Team Rocket's X",
  "Marnie's X", "Iono's X", ...) appears on **17 distinct real tags**.
* The "damage scales with same-tag board count" attack pattern matches
  **12 real attacks** across the pool.

So the fix is two small, purely text/name-derived additions to the
existing, already-general `Role`-tagging system (`ptcg/core/carddb.py`,
which by design already extracts every other judgement from rules text, not
hardcoded card IDs) -- a `Card.affiliation` field parsed from the card's own
name, and two new pattern-derived roles (`BOARD_SCALING`, `BOARD_GATED`).
The heuristic then gets a general rule: *building board presence of a tag
you already have a synergy card for is worth something*, with zero
reference to "Team Rocket" anywhere in the code.

In [ ]:
week6_bench = json.load(open("artifacts/week6_archetype_bench.json"))
for du in week6_bench["duels"]:
    names = {du["a"], du["b"]}
    if "rocket_mewtwo_heuristic" in names or "rocket_mewtwo_bc" in names:
        print(du["a"], "vs", du["b"], "-> a_winrate", du["a_winrate"], "ci95", du["ci95"])


**Result, and it's an honest negative one**: the fix works exactly as
designed -- traced a real game and confirmed the heuristic now builds up to
6 Team Rocket's Pokemon in play (well past Mewtwo ex's 4-needed threshold),
something it structurally could not reason about before. But this did
**not** translate into a meaningful win-rate improvement: `rocket_mewtwo`
piloted by the fixed heuristic still loses to almost everything (2.7% vs
the current best lineup, CI [1.0%, 6.7%] -- overlapping the pre-fix CI, not
a statistically distinguishable improvement).

**What this tells us, precisely**: board-count synergy was real but not the
whole story. This deck also requires sequencing two different attackers and
managing two energy types, and a hand-tuned heuristic whose other scoring
constants were implicitly validated against a simple single-attacker deck
struggles with that regardless of the specific synergy fix. Rather than
keep bolting on deck-specific patches (which is exactly the overfitting
this session was trying to avoid), the conclusion drawn was: **the reliable
lever for this deck is the trained network, not the rule-based heuristic**
-- confirmed independently twice now (`bc_rocket_mewtwo` beat the heuristic
99.3% and 99.99%+ across both benches).

## 11. Closing the gap for real: self-play, SPR, and population deck search

Section 9 left one lever unpulled: the network alone (`bc_rocket_mewtwo`,
pure imitation from real replay data) was strong relative to the heuristic
but still lost 33.3% to the current submission as a whole package. Pure
imitation learning has a hard ceiling -- it can only ever approach its
training data's own skill level, never exceed it. Closing that gap needs
techniques that let the policy improve *past* what it imitated:

* **Self-play refinement**, reusing this project's own proven Week-2
  machinery (PFSP opponent sampling, frozen-teacher KL regularization,
  temperature annealing) -- built once, generically, and reused here
  unchanged for a different deck, exactly the "robust architecture" the
  fix was aimed at.
* **SPR (self-predictive representations)**, a self-supervised auxiliary
  loss: predict the trunk embedding of your own *next* decision from the
  current one plus the action just taken, trained against a stop-gradient
  target (BYOL-style, so it needs no negative samples / large batch --
  what makes it viable on a CPU-only machine). The idea: a representation
  that can predict its own future is forced to encode more of the game's
  actual structure, which should make the *policy* head's job easier too.
* **MAP-Elites-lite + Double-Oracle-lite**: rather than committing to one
  archetype (`rocket_mewtwo`) because it happened to be the most-represented
  deck in one data pull, build a real behavior-descriptor archive across
  all the real archetypes captured this session, and solve for an
  approximate Nash mixture over the survivors via replicator dynamics --
  the honest, scoped version of the population-search + meta-game-solving
  approach the original planning documents specified.

In [ ]:
sp = json.load(open("artifacts/week7_selfplay_report.json"))
final_round = sp["history"][-1]
print("final round deterministic checks:", final_round["deterministic_check"])


**Result**: after 3 rounds of PFSP self-play seeded from
`bc_rocket_mewtwo`, the refined checkpoint beats its own frozen teacher
**65.0%** of the time (CI [52.4%, 75.8%]) -- a real, statistically
significant improvement, not noise. Fuzzed clean at the Week-0 standard
(20,000 observations + 150 games, 0 crashes, 0 illegal actions) before
being considered for anything further.

In [ ]:
bench = json.load(open("artifacts/week7_selfplay_bench.json"))
for du in bench["duels"]:
    names = {du["a"], du["b"]}
    if "rocket_mewtwo_bc" in names:
        print(du["a"], "vs", du["b"], "-> a_winrate", du["a_winrate"], "ci95", du["ci95"])


**Result, honestly**: the self-play-refined checkpoint's improvement
did **not** transfer to the matchup that actually matters. Against the
current submission's lineup it scored 29.3% vs `lucario_fighting` + heuristic,
33.3% vs + bc_text, 30.0% vs + router -- essentially unchanged from before
self-play (33.3% across the board). It still solidly beats the Crustle wall
(99.3%, even better than before).

**Why, in retrospect**: the self-play league only ever played `rocket_mewtwo`
against opponents piloting the *same* deck (itself, past checkpoints, generic
baselines) -- it never once played against `lucario_fighting`. Improving
relative to your own lineage doesn't guarantee improving against a
genuinely different competitor that was never part of training. This is a
real, useful architectural finding: self-play closes gaps *within* a
training population, not automatically against arbitrary external
opponents -- worth remembering before assuming any self-play result
transfers to the actual target.

## 13. The real fix: search a population instead of committing to one deck

Section 9's honest failure and Section 11's honest failure share a root
cause: everything after the archetype pivot assumed `rocket_mewtwo` -- the
most-represented real deck in one identity-filtered data pull -- was the
right deck to commit to, and then tried to make it work harder (a better
heuristic, a better-trained network, self-play refinement). None of that
questioned the choice of deck itself.

**MAP-Elites + Double-Oracle, built for real this pass** (not the bounded
local search from Section 3, a genuine population archive + game-theoretic
solve): seed a behavior-descriptor archive with all 5 real archetypes
captured from the live ladder this session (`rocket_mewtwo`,
`lucario_fighting`, and three more pulled specifically for this search --
Marnie's Grimmsnarl ex, Mega Kangaskhan ex, and Dragapult ex, each the
highest-real-leaderboard-rank example available), mutate via the same
legal-swap machinery every deck search this session has used, keep the
best-worst-case deck per behavior cell, then solve for an approximate Nash
equilibrium over the survivors via replicator dynamics.

In [ ]:
archive = json.load(open("artifacts/week7_map_elites.json"))["archive"]
for entry in archive[:6]:
    print(f"{entry['name']:24s} worst={entry['worst']:.3f} avg={entry['avg']:.3f}  {entry['results']}")

do = json.load(open("artifacts/week7_double_oracle.json"))
print()
print("Double-Oracle equilibrium mixture:")
for m in do["equilibrium_mixture"][:4]:
    print(" ", m)


**Result**: the search surfaced something none of Sections 8-12 found by
iterating on `rocket_mewtwo` -- a *different* real archetype,
**Mega Kangaskhan ex**, with a dramatically better worst-case floor (41.3%
vs the Crustle wall, using nothing but the existing, unmodified heuristic)
than `rocket_mewtwo` ever achieved (0% worst-case against `lucario_fighting`
even after every fix tried). The Double-Oracle equilibrium puts roughly 78%
combined weight on Kangaskhan-lineage decks and **zero** on every Dragapult
variant and on `rocket_mewtwo` itself -- a real, quantified signal, not a
guess, that the archetype choice mattered more than any of the tuning work
that followed it.

**A genuinely interesting design detail the search surfaced**: the specific
real Kangaskhan decklist (from the same rank-3 player, "LiamK", who also
piloted `rocket_mewtwo`) runs **Crustle itself**, 4 copies, alongside Mega
Kangaskhan ex. Rather than avoiding the ex-counter problem this entire
project chased since Section 3, this real player's deck answers it directly
-- it plays both an ex win condition *and* a hard counter to what counters
ex win conditions, in the same 60 cards. That is not a technique this
session's own deck-searches (which only ever locally patch one deck) would
ever have found; it took evaluating a real, independently-built deck to see
it.

In [ ]:
bench = json.load(open("artifacts/week7_kangaskhan_bench.json"))
for du in bench["duels"]:
    names = {du["a"], du["b"]}
    if "kangaskhan_crustle_heuristic" in names:
        print(du["a"], "vs", du["b"], "-> a_winrate", du["a_winrate"], "ci95", du["ci95"])


**Confirmed at full rigor (150 games/pair, Wilson CIs) -- the first
real, statistically decisive win over the actual live submission found in
this entire project**:

| Matchup | Win rate | 95% CI |
|---|---|---|
| vs `lucario_fighting` + heuristic (what is actually live) | **60.7%** | [52.7%, 68.1%] |
| vs `lucario_fighting` + router | **72.0%** | [64.3%, 78.6%] |
| vs `lucario_fighting` + bc_text | 57.3% | [49.3%, 65.0%] |

Both CIs against heuristic and router exclude 50% -- a real, not-noise
result. Fuzzed clean at the Week-0 standard (20,000 hostile observations,
0 crashes, 0 illegal actions) and self-verified through an isolated
`submission.tar.gz` rebuild before being treated as a candidate.

**The strategic lesson, stated plainly**: every technique built this
session (imitation learning, distillation, routing, self-play, SPR) is real
and works as designed, but none of it beat the actual competitive target as
reliably as simply *evaluating more real alternatives before committing* --
population search over evidence-backed candidates, not deeper investment in
the first reasonable-looking one.

## 14. Conclusions

The throughline across every section here is the same: a metric that looks good
in isolation (training accuracy, average win rate, a single strong matchup) is
not the same thing as a robust, competition-ready agent. Card-text embeddings
beat one-hot IDs specifically because they generalize past the training
distribution. A deck that wins on average can still have a near-guaranteed loss
against a real, legal counter-deck. A model with better validation loss can play
worse. A single trusted policy is more fragile than one that knows when to
defer. None of it matters if the agent can be crashed or tricked into an
illegal move by an input nobody tested against.

Every technique here is implemented in the companion Simulation-track
submission: text-embedding card representations, a bounded deck-robustness
search, an efficiency-optimized distilled policy, confidence-gated routing, a
20,000-case adversarial fuzz gate, and real-replay imitation learning from the
live ladder's strongest players.